# Phase 9 — Hopsworks MLOps Pipeline

- **9A.** Inspect repository and lock reuse contract
- **9B.** Secure Hopsworks connectivity and dry-run mode
- **9C.** Create three minimal feature groups
- **9D.** Dry-run and perform historical backfill
- **9E.** Create feature view and validate training-dataset parity
- **9F.** Register the existing approved model as champion
- **9G.** Connect Phase 5 model loading to registry with fallback
- **9H.** Build incremental feature updates
- **9I.** Build controlled retraining cycle
- **9J.** Add promotion and rollback protection
- **9K.** Add focused tests and final Phase 9 report

## **9A.** Repository Inspection and Reuse Contract

The purpose of Phase 9A is to identify the existing canonical-data,
feature-engineering, model-training, and inference components that must be
reused by the MLOps pipelines.

No Hopsworks resources are created and no production code is changed during
this subphase.

In [1]:
from pathlib import Path
import ast
import hashlib
import json
from datetime import datetime, timezone

import joblib
import pandas as pd


PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (
    PROJECT_ROOT / "notebooks"
).exists(), "Could not resolve the project root."

print("Project root:", PROJECT_ROOT)

Project root: /home/riyan/Riyan/projects/pearls-aqi-predictor


In [2]:
SEARCH_DIRECTORIES = [
    "app",
    "data",
    "models",
    "inference",
    "aqi",
    "dashboard",
    "tests",
    "reports",
]

repository_files = []

for directory_name in SEARCH_DIRECTORIES:
    directory = PROJECT_ROOT / directory_name

    if not directory.exists():
        continue

    repository_files.extend(
        path.relative_to(PROJECT_ROOT).as_posix()
        for path in directory.rglob("*")
        if path.is_file()
    )

repository_files = sorted(repository_files)

print("Repository files found:", len(repository_files))

for path in repository_files:
    print(path)

Repository files found: 280
app/__init__.py
app/__pycache__/__init__.cpython-312.pyc
app/alerts/__init__.py
app/alerts/__pycache__/__init__.cpython-312.pyc
app/alerts/__pycache__/aqi_alerts.cpython-312.pyc
app/alerts/aqi_alerts.py
app/api/__init__.py
app/api/__pycache__/__init__.cpython-312.pyc
app/api/__pycache__/config.cpython-312.pyc
app/api/__pycache__/dependencies.cpython-312.pyc
app/api/__pycache__/errors.cpython-312.pyc
app/api/__pycache__/main.cpython-312.pyc
app/api/__pycache__/middleware.cpython-312.pyc
app/api/config.py
app/api/dependencies.py
app/api/errors.py
app/api/main.py
app/api/middleware.py
app/api/routes/__init__.py
app/api/routes/__pycache__/__init__.cpython-312.pyc
app/api/routes/__pycache__/alerts.cpython-312.pyc
app/api/routes/__pycache__/forecast.cpython-312.pyc
app/api/routes/__pycache__/health.cpython-312.pyc
app/api/routes/__pycache__/metadata.cpython-312.pyc
app/api/routes/alerts.py
app/api/routes/forecast.py
app/api/routes/health.py
app/api/routes/metadata

### Phase 2 and Phase 3 artifact discovery

The approved local artifacts will be used as the source of truth for the first
Hopsworks migration.

They support:

- feature-contract validation
- local-versus-Hopsworks dataset parity
- initial champion registration
- model checksum validation
- production-model resolution testing

In [3]:
FEATURE_COLUMNS_PATH = (
    PROJECT_ROOT
    / "data"
    / "training"
    / "feature_columns.json"
)

TRAINING_DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "training"
    / "feature_dataset_full.parquet"
)

TRAIN_DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "training"
    / "train_dataset.parquet"
)

VALIDATION_DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "training"
    / "validation_dataset.parquet"
)

TEST_DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "training"
    / "test_dataset.parquet"
)

PHASE_2_VALIDATION_REPORT_PATH = (
    PROJECT_ROOT
    / "data"
    / "training"
    / "phase_2_validation_report.json"
)

MODEL_ARTIFACT_PATH = (
    PROJECT_ROOT
    / "models"
    / "best_model.joblib"
)

MODEL_FEATURE_COLUMNS_PATH = (
    PROJECT_ROOT
    / "models"
    / "model_feature_columns.json"
)

MODEL_METADATA_PATH = (
    PROJECT_ROOT
    / "models"
    / "model_metadata.json"
)

MODEL_SELECTION_REPORT_PATH = (
    PROJECT_ROOT
    / "models"
    / "model_selection_report.json"
)


required_artifacts = {
    "feature_columns": FEATURE_COLUMNS_PATH,
    "training_dataset": TRAINING_DATASET_PATH,
    "train_dataset": TRAIN_DATASET_PATH,
    "validation_dataset": VALIDATION_DATASET_PATH,
    "test_dataset": TEST_DATASET_PATH,
    "phase_2_validation_report": (
        PHASE_2_VALIDATION_REPORT_PATH
    ),
    "model_artifact": MODEL_ARTIFACT_PATH,
    "model_feature_columns": (
        MODEL_FEATURE_COLUMNS_PATH
    ),
    "model_metadata": MODEL_METADATA_PATH,
    "model_selection_report": (
        MODEL_SELECTION_REPORT_PATH
    ),
}

artifact_existence = {
    name: path.exists()
    for name, path in required_artifacts.items()
}

artifact_existence

{'feature_columns': True,
 'training_dataset': True,
 'train_dataset': True,
 'validation_dataset': True,
 'test_dataset': True,
 'phase_2_validation_report': True,
 'model_artifact': True,
 'model_feature_columns': True,
 'model_metadata': True,
 'model_selection_report': True}

### Reusable Python functions

The following inspection searches the application and inference packages for
functions related to:

- PM2.5 cleaning and validation
- weather processing
- canonical dataset construction
- lag and rolling features
- horizon expansion
- target construction
- chronological splitting
- feature-order enforcement
- inference feature generation
- model loading

The final reuse decision must be based on the function implementation, not
only its name.

In [4]:
FUNCTION_KEYWORDS = {
    "canonical",
    "feature",
    "pm25",
    "weather",
    "lag",
    "rolling",
    "horizon",
    "target",
    "split",
    "inference",
    "model",
    "load",
    "validate",
    "normalize",
    "clean",
}


PYTHON_SEARCH_DIRECTORIES = [
    PROJECT_ROOT / "app",
    PROJECT_ROOT / "inference",
    PROJECT_ROOT / "aqi",
]


discovered_functions = []

for search_directory in PYTHON_SEARCH_DIRECTORIES:
    if not search_directory.exists():
        continue

    for python_file in search_directory.rglob("*.py"):
        source = python_file.read_text(
            encoding="utf-8"
        )

        try:
            tree = ast.parse(
                source,
                filename=str(python_file),
            )
        except SyntaxError as error:
            print(
                "Skipped invalid Python file:",
                python_file,
                error,
            )
            continue

        for node in ast.walk(tree):
            if not isinstance(
                node,
                (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                ),
            ):
                continue

            function_name = node.name.lower()

            if any(
                keyword in function_name
                for keyword in FUNCTION_KEYWORDS
            ):
                discovered_functions.append(
                    {
                        "file": (
                            python_file.relative_to(
                                PROJECT_ROOT
                            ).as_posix()
                        ),
                        "function": node.name,
                        "line": node.lineno,
                    }
                )


discovered_functions = sorted(
    discovered_functions,
    key=lambda item: (
        item["file"],
        item["line"],
    ),
)

print(
    "Potential reusable functions:",
    len(discovered_functions),
)

for item in discovered_functions:
    print(
        f"{item['file']}:{item['line']} "
        f"{item['function']}"
    )

Potential reusable functions: 70
app/alerts/aqi_alerts.py:143 _validate_enriched_forecast
app/api/config.py:98 validate_api_prefix
app/api/config.py:115 normalize_environment
app/api/config.py:125 normalize_log_level
app/api/services/artifact_repository.py:152 load_latest
app/api/services/artifact_repository.py:222 _validate_required_files_exist
app/api/services/artifact_repository.py:276 _load_and_validate_bundle
app/api/services/artifact_repository.py:394 _load_forecast
app/api/services/artifact_repository.py:417 _load_json
app/api/services/artifact_repository.py:451 _validate_forecast
app/api/services/artifact_repository.py:627 _validate_hourly_timestamps
app/api/services/artifact_repository.py:673 _validate_rolling_fields
app/api/services/artifact_repository.py:711 _validate_phase_6_status
app/api/services/artifact_repository.py:734 _validate_run_consistency
app/aqi/forecast_enrichment.py:18 _normalize_pm25_timeline
app/aqi/forecast_enrichment.py:157 _calculate_rolling_24h_metrics


### Ordered feature-contract validation

The Phase 2 training feature contract and Phase 3 model feature contract must
match exactly.

The comparison checks:

- feature count
- feature names
- missing or additional features
- feature order

Feature order is important because the trained model expects columns in the
same order used during training.

In [5]:
def load_json_list(
    path: Path,
) -> list[str]:
    payload = json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )

    if isinstance(payload, list):
        return [
            str(value)
            for value in payload
        ]

    if isinstance(payload, dict):
        for candidate_key in (
            "feature_columns",
            "model_feature_columns",
            "features",
            "columns",
        ):
            candidate = payload.get(
                candidate_key
            )

            if isinstance(candidate, list):
                return [
                    str(value)
                    for value in candidate
                ]

    raise ValueError(
        f"Could not find a feature-column list in {path}"
    )


feature_columns = load_json_list(
    FEATURE_COLUMNS_PATH
)

model_feature_columns = load_json_list(
    MODEL_FEATURE_COLUMNS_PATH
)


feature_contract_summary = {
    "feature_columns_count": len(
        feature_columns
    ),
    "model_feature_columns_count": len(
        model_feature_columns
    ),
    "same_features": (
        set(feature_columns)
        == set(model_feature_columns)
    ),
    "same_order": (
        feature_columns
        == model_feature_columns
    ),
    "only_in_feature_columns": sorted(
        set(feature_columns)
        - set(model_feature_columns)
    ),
    "only_in_model_feature_columns": sorted(
        set(model_feature_columns)
        - set(feature_columns)
    ),
}

feature_contract_summary

{'feature_columns_count': 56,
 'model_feature_columns_count': 56,
 'same_features': True,
 'same_order': True,
 'only_in_feature_columns': [],
 'only_in_model_feature_columns': []}

### Approved Phase 2 training dataset

`feature_dataset_full.parquet` is the complete horizon-expanded dataset created
during Phase 2.

The inspection records:

- dataset dimensions
- reference-time range
- target-time range
- forecast horizons
- duplicate training keys
- missing values
- target column
- chronological split artifacts

This dataset will later be compared with the first training dataset generated
from Hopsworks.

In [6]:
training_df = pd.read_parquet(
    TRAINING_DATASET_PATH
)

print("Shape:", training_df.shape)
print()
print("Columns:")

for column in training_df.columns:
    print(column)

Shape: (512867, 59)

Columns:
reference_time
target_time
pm25_current
pm25_lag_1h
pm25_lag_3h
pm25_lag_6h
pm25_lag_12h
pm25_lag_24h
pm25_mean_3h
pm25_mean_6h
pm25_mean_12h
pm25_mean_24h
pm25_change_1h
pm25_change_6h
pm25_change_24h
temperature_2m
relative_humidity_2m
dew_point_2m
surface_pressure
precipitation
rain
cloud_cover
wind_speed_10m
wind_direction_10m
wind_gusts_10m
wind_direction_10m_sin
wind_direction_10m_cos
reference_hour
reference_day_of_week
reference_month
reference_hour_sin
reference_hour_cos
reference_day_of_week_sin
reference_day_of_week_cos
reference_month_sin
reference_month_cos
forecast_horizon_hours
target_temperature_2m
target_relative_humidity_2m
target_dew_point_2m
target_surface_pressure
target_precipitation
target_rain
target_cloud_cover
target_wind_speed_10m
target_wind_direction_10m
target_wind_gusts_10m
target_wind_direction_10m_sin
target_wind_direction_10m_cos
target_hour
target_day_of_week
target_month
target_hour_sin
target_hour_cos
target_day_of_week

In [7]:
def resolve_column(
    dataframe: pd.DataFrame,
    candidates: list[str],
) -> str | None:
    for candidate in candidates:
        if candidate in dataframe.columns:
            return candidate

    return None


REFERENCE_TIME_COLUMN = resolve_column(
    training_df,
    [
        "reference_time",
        "reference_time_utc",
        "datetime_utc",
    ],
)

TARGET_TIME_COLUMN = resolve_column(
    training_df,
    [
        "target_time",
        "target_time_utc",
    ],
)

HORIZON_COLUMN = resolve_column(
    training_df,
    [
        "forecast_horizon_hours",
        "horizon_hours",
        "horizon",
    ],
)

TARGET_COLUMN = resolve_column(
    training_df,
    [
        "target_pm25_ug_m3",
        "target_pm25",
    ],
)


resolved_training_columns = {
    "reference_time": REFERENCE_TIME_COLUMN,
    "target_time": TARGET_TIME_COLUMN,
    "horizon": HORIZON_COLUMN,
    "target": TARGET_COLUMN,
}

resolved_training_columns

{'reference_time': 'reference_time',
 'target_time': 'target_time',
 'horizon': 'forecast_horizon_hours',
 'target': 'target_pm25_ug_m3'}

In [8]:
training_df[
    REFERENCE_TIME_COLUMN
] = pd.to_datetime(
    training_df[
        REFERENCE_TIME_COLUMN
    ],
    utc=True,
)

training_df[
    TARGET_TIME_COLUMN
] = pd.to_datetime(
    training_df[
        TARGET_TIME_COLUMN
    ],
    utc=True,
)


duplicate_key_count = int(
    training_df.duplicated(
        subset=[
            REFERENCE_TIME_COLUMN,
            HORIZON_COLUMN,
        ]
    ).sum()
)


training_dataset_summary = {
    "path": (
        TRAINING_DATASET_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix()
    ),
    "row_count": int(
        len(training_df)
    ),
    "column_count": int(
        len(training_df.columns)
    ),
    "reference_time_column": (
        REFERENCE_TIME_COLUMN
    ),
    "reference_start": (
        training_df[
            REFERENCE_TIME_COLUMN
        ]
        .min()
        .isoformat()
    ),
    "reference_end": (
        training_df[
            REFERENCE_TIME_COLUMN
        ]
        .max()
        .isoformat()
    ),
    "target_time_column": (
        TARGET_TIME_COLUMN
    ),
    "target_start": (
        training_df[
            TARGET_TIME_COLUMN
        ]
        .min()
        .isoformat()
    ),
    "target_end": (
        training_df[
            TARGET_TIME_COLUMN
        ]
        .max()
        .isoformat()
    ),
    "horizon_column": HORIZON_COLUMN,
    "horizon_min": int(
        training_df[
            HORIZON_COLUMN
        ].min()
    ),
    "horizon_max": int(
        training_df[
            HORIZON_COLUMN
        ].max()
    ),
    "target_column": TARGET_COLUMN,
    "duplicate_key_count": (
        duplicate_key_count
    ),
    "total_missing_values": int(
        training_df.isna().sum().sum()
    ),
}

training_dataset_summary

{'path': 'data/training/feature_dataset_full.parquet',
 'row_count': 512867,
 'column_count': 59,
 'reference_time_column': 'reference_time',
 'reference_start': '2025-07-09T00:00:00+00:00',
 'reference_end': '2026-07-23T22:00:00+00:00',
 'target_time_column': 'target_time',
 'target_start': '2025-07-09T01:00:00+00:00',
 'target_end': '2026-07-23T23:00:00+00:00',
 'horizon_column': 'forecast_horizon_hours',
 'horizon_min': 1,
 'horizon_max': 72,
 'target_column': 'target_pm25_ug_m3',
 'duplicate_key_count': 0,
 'total_missing_values': 0}

In [9]:
split_paths = {
    "train": TRAIN_DATASET_PATH,
    "validation": VALIDATION_DATASET_PATH,
    "test": TEST_DATASET_PATH,
}

split_summaries = {}

for split_name, split_path in split_paths.items():
    split_df = pd.read_parquet(
        split_path
    )

    split_df[
        REFERENCE_TIME_COLUMN
    ] = pd.to_datetime(
        split_df[
            REFERENCE_TIME_COLUMN
        ],
        utc=True,
    )

    split_summaries[split_name] = {
        "path": split_path.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "rows": int(len(split_df)),
        "reference_start": (
            split_df[
                REFERENCE_TIME_COLUMN
            ]
            .min()
            .isoformat()
        ),
        "reference_end": (
            split_df[
                REFERENCE_TIME_COLUMN
            ]
            .max()
            .isoformat()
        ),
    }

split_summaries

{'train': {'path': 'data/training/train_dataset.parquet',
  'rows': 364798,
  'reference_start': '2025-07-09T00:00:00+00:00',
  'reference_end': '2026-03-18T11:00:00+00:00'},
 'validation': {'path': 'data/training/validation_dataset.parquet',
  'rows': 71256,
  'reference_start': '2026-03-21T12:00:00+00:00',
  'reference_end': '2026-05-27T21:00:00+00:00'},
 'test': {'path': 'data/training/test_dataset.parquet',
  'rows': 76813,
  'reference_start': '2026-05-30T22:00:00+00:00',
  'reference_end': '2026-07-23T22:00:00+00:00'}}

### Approved Phase 3 model

The selected Phase 3 model is loaded to verify that the artifact remains
readable in the current environment.

A SHA-256 checksum is generated so that the same artifact can later be:

- registered as the initial Hopsworks champion
- downloaded and validated
- cached safely
- traced from a production forecast

In [10]:
def calculate_sha256(
    path: Path,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


approved_model = joblib.load(
    MODEL_ARTIFACT_PATH
)


approved_model_summary = {
    "path": (
        MODEL_ARTIFACT_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix()
    ),
    "model_type": (
        f"{type(approved_model).__module__}."
        f"{type(approved_model).__name__}"
    ),
    "checksum_sha256": calculate_sha256(
        MODEL_ARTIFACT_PATH
    ),
    "loaded_successfully": True,
    "metadata_path": (
        MODEL_METADATA_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix()
    ),
    "selection_report_path": (
        MODEL_SELECTION_REPORT_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix()
    ),
}

approved_model_summary

{'path': 'models/best_model.joblib',
 'model_type': 'xgboost.sklearn.XGBRegressor',
 'checksum_sha256': '9d1cbb032a6f376892c2ad047bbf93bf485d18b21b47d12872a2c367073b5a49',
 'loaded_successfully': True,
 'metadata_path': 'models/model_metadata.json',
 'selection_report_path': 'models/model_selection_report.json'}

In [11]:
model_metadata = json.loads(
    MODEL_METADATA_PATH.read_text(
        encoding="utf-8"
    )
)

model_selection_report = json.loads(
    MODEL_SELECTION_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)

print("Model metadata keys:")
print(sorted(model_metadata.keys()))

print()
print("Model selection report keys:")
print(sorted(model_selection_report.keys()))

Model metadata keys:
['artifact_paths', 'best_iteration', 'data_ranges', 'final_test_metrics', 'forecast_description', 'horizon_groups', 'input_feature_count', 'limitations', 'model_name', 'model_parameters', 'model_type', 'ordered_feature_names', 'project_name', 'random_seed', 'routing', 'row_counts', 'selected_strategy', 'software_versions', 'target_column', 'test_baseline_metrics', 'training_date_utc', 'validation_metrics']

Model selection report keys:
['final_untouched_test_metrics', 'limitations', 'major_error_patterns', 'models_evaluated', 'next_phase_recommendation', 'overfitting_assessment', 'selected_model', 'selected_strategy', 'selection_reason', 'test_improvement_over_current_persistence', 'validation_metrics', 'xgboost_configurations']


### Phase 5 live-inference boundary

Phase 9 must modify only the model-loading boundary of live inference.

The following components must be identified before Hopsworks integration:

- live feature-state reconstruction
- 72-horizon inference-row construction
- ordered feature selection
- feature-contract validation
- local model loading
- model prediction
- operational clipping
- inference metadata generation

Feature calculations must remain unchanged.

In [12]:
INFERENCE_KEYWORDS = {
    "inference",
    "feature",
    "model",
    "load",
    "predict",
    "horizon",
    "contract",
    "column",
    "matrix",
}


inference_functions = [
    item
    for item in discovered_functions
    if (
        item["file"].startswith(
            "inference/"
        )
        or "inference" in item["file"]
        or any(
            keyword in item["function"].lower()
            for keyword in INFERENCE_KEYWORDS
        )
    )
]

print(
    "Potential inference functions:",
    len(inference_functions),
)

for item in inference_functions:
    print(
        f"{item['file']}:{item['line']} "
        f"{item['function']}"
    )

Potential inference functions: 34
app/api/services/artifact_repository.py:152 load_latest
app/api/services/artifact_repository.py:276 _load_and_validate_bundle
app/api/services/artifact_repository.py:394 _load_forecast
app/api/services/artifact_repository.py:417 _load_json
app/core/config.py:91 best_model_path
app/core/config.py:96 model_feature_contract_path
app/core/config.py:101 model_metadata_path
app/core/config.py:106 model_selection_report_path
app/core/config.py:111 phase_2_feature_contract_path
app/features/live_feature_builder.py:84 add_pm25_history_features
app/features/live_feature_builder.py:129 add_wind_direction_features
app/features/live_feature_builder.py:154 add_time_features
app/features/live_feature_builder.py:219 build_reference_feature_table
app/features/live_feature_builder.py:275 build_target_weather_feature_table
app/features/live_feature_builder.py:337 build_feature_rows
app/inference/predictor.py:36 _load_json
app/inference/predictor.py:60 _validate_required_

### Phase 9 reuse contract

The Hopsworks integration will reuse existing project logic as follows:

| Existing component | Phase 9 responsibility |
|---|---|
| PM2.5 validation | Historical and incremental ingestion |
| Weather validation | Historical and incremental ingestion |
| Canonical hourly builder | Feature-store backfill preparation |
| Reference-time feature builder | Engineered feature-group generation |
| Horizon expansion | Training-dataset generation |
| Target construction | Training-dataset generation |
| Chronological split logic | Automated candidate training |
| Ordered feature contract | Dataset parity and model validation |
| Phase 5 inference builder | Candidate and production smoke tests |
| Current model loader | Local fallback |

Hopsworks adapter modules will manage storage, feature views, training datasets,
and model registry operations. They will not contain independent feature
formulas.

In [13]:
REPORT_DIRECTORY = (
    PROJECT_ROOT
    / "reports"
    / "phase_9"
)

REPORT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


repository_integration_report = {
    "phase": "9A",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": (
        "REPOSITORY_INSPECTION_IN_PROGRESS"
    ),
    "canonical_pipeline": {
        "files": [],
        "reusable_functions": [],
    },
    "feature_engineering": {
        "files": [],
        "reusable_functions": [],
        "feature_columns_path": (
            FEATURE_COLUMNS_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix()
        ),
        "model_feature_columns_path": (
            MODEL_FEATURE_COLUMNS_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix()
        ),
        "feature_columns_count": len(
            feature_columns
        ),
        "model_feature_columns_count": len(
            model_feature_columns
        ),
        "same_features": (
            set(feature_columns)
            == set(model_feature_columns)
        ),
        "same_order": (
            feature_columns
            == model_feature_columns
        ),
    },
    "training_dataset": {
        **training_dataset_summary,
        "splits": split_summaries,
        "phase_2_validation_report_path": (
            PHASE_2_VALIDATION_REPORT_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix()
        ),
    },
    "approved_model": {
        **approved_model_summary,
        "feature_columns_path": (
            MODEL_FEATURE_COLUMNS_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix()
        ),
    },
    "live_inference": {
        "feature_builder_file": "",
        "feature_builder_function": "",
        "model_loader_file": "",
        "model_loader_function": "",
        "feature_contract_validation": False,
        "expected_prediction_rows": 72,
    },
    "reuse_decisions": [],
    "small_changes_required": [],
    "duplicate_implementations_detected": [],
    "phase_9a_approved": False,
}

repository_integration_report

{'phase': '9A',
 'generated_at_utc': '2026-07-30T09:46:14.027006+00:00',
 'status': 'REPOSITORY_INSPECTION_IN_PROGRESS',
 'canonical_pipeline': {'files': [], 'reusable_functions': []},
 'feature_engineering': {'files': [],
  'reusable_functions': [],
  'feature_columns_path': 'data/training/feature_columns.json',
  'model_feature_columns_path': 'models/model_feature_columns.json',
  'feature_columns_count': 56,
  'model_feature_columns_count': 56,
  'same_features': True,
  'same_order': True},
 'training_dataset': {'path': 'data/training/feature_dataset_full.parquet',
  'row_count': 512867,
  'column_count': 59,
  'reference_time_column': 'reference_time',
  'reference_start': '2025-07-09T00:00:00+00:00',
  'reference_end': '2026-07-23T22:00:00+00:00',
  'target_time_column': 'target_time',
  'target_start': '2025-07-09T01:00:00+00:00',
  'target_end': '2026-07-23T23:00:00+00:00',
  'horizon_column': 'forecast_horizon_hours',
  'horizon_min': 1,
  'horizon_max': 72,
  'target_column':

### Manual inspection results

The following report fields must be completed after reviewing the discovered
functions:

- canonical pipeline files
- source-validation functions
- canonical builder function
- feature-engineering files and functions
- horizon-expansion function
- target-construction function
- chronological-split function
- live inference feature-builder function
- current model-loader function
- feature-contract validation behavior
- required small integration changes
- duplicate implementations, if any

Phase 9A should not be approved while any critical function remains
unidentified.

In [14]:
repository_integration_report[
    "canonical_pipeline"
] = {
    "files": [
        "notebooks/02_build_canonical_dataset.ipynb",
        "app/data_sources/openaq_client.py",
        "app/data_sources/open_meteo_client.py",
        "app/data/validation.py",
    ],
    "reusable_functions": [
        "_normalize_hourly_results",
        "_normalize_hourly_weather",
        "_validate_hourly_weather",
        "_normalize_utc_timestamp",
    ],
    "notebook_defined_logic": [
        "PM2.5 quality-rule application",
        "duplicate-hour handling",
        "canonical hourly timestamp construction",
        "PM2.5 and weather merge",
        "canonical dataset validation",
    ],
}


repository_integration_report[
    "feature_engineering"
][
    "files"
] = [
    "app/features/live_feature_builder.py",
    "notebooks/03_build_training_dataset.ipynb",
]

repository_integration_report[
    "feature_engineering"
][
    "reusable_functions"
] = [
    "validate_hourly_timeline",
    "add_pm25_history_features",
    "add_wind_direction_features",
    "add_time_features",
    "build_reference_feature_table",
    "build_target_weather_feature_table",
    "build_feature_rows",
]

repository_integration_report[
    "feature_engineering"
][
    "notebook_defined_logic"
] = [
    "historical horizon expansion for horizons 1 through 72",
    "target_time construction",
    "target_pm25_ug_m3 construction",
    "training-row validity filtering",
    "leakage validation",
    "chronological train, validation, and test splitting",
]


repository_integration_report[
    "live_inference"
].update(
    {
        "feature_builder_file": (
            "app/features/live_feature_builder.py"
        ),
        "feature_builder_function": (
            "build_feature_rows"
        ),
        "model_loader_file": (
            "app/inference/predictor.py"
        ),
        "model_loader_function": (
            "load_model_artifacts"
        ),
        "feature_contract_validation": True,
    }
)


repository_integration_report[
    "reuse_decisions"
] = [
    (
        "Reuse OpenAQ hourly normalization from "
        "app/data_sources/openaq_client.py."
    ),
    (
        "Reuse Open-Meteo normalization and validation from "
        "app/data_sources/open_meteo_client.py."
    ),
    (
        "Reuse UTC timestamp normalization from "
        "app/data/validation.py."
    ),
    (
        "Preserve the canonical dataset behavior currently "
        "implemented in notebooks/02_build_canonical_dataset.ipynb."
    ),
    (
        "Reuse app/features/live_feature_builder.py for PM2.5 "
        "history, wind-direction, time, reference-time, "
        "target-weather, and inference-row features."
    ),
    (
        "Preserve the training-dataset behavior currently "
        "implemented in notebooks/03_build_training_dataset.ipynb."
    ),
    (
        "Reuse load_model_artifacts from "
        "app/inference/predictor.py."
    ),
    (
        "Reuse the strict Phase 2 and model feature-contract "
        "comparison before prediction."
    ),
    (
        "Change only the model-loading boundary when "
        "Hopsworks Model Registry support is introduced."
    ),
]


repository_integration_report[
    "small_changes_required"
] = [
    (
        "Extract canonical dataset construction from "
        "notebooks/02_build_canonical_dataset.ipynb into a "
        "reusable Python module without changing behavior."
    ),
    (
        "Extract horizon expansion, target construction, "
        "leakage checks, and chronological splitting from "
        "notebooks/03_build_training_dataset.ipynb into a "
        "reusable Python module without changing behavior."
    ),
    (
        "Add a configurable model loader supporting "
        "LOCAL_ARTIFACT and HOPSWORKS_REGISTRY modes."
    ),
]


repository_integration_report[
    "duplicate_implementations_detected"
] = []


repository_integration_report

{'phase': '9A',
 'generated_at_utc': '2026-07-30T09:46:14.027006+00:00',
 'status': 'REPOSITORY_INSPECTION_IN_PROGRESS',
 'canonical_pipeline': {'files': ['notebooks/02_build_canonical_dataset.ipynb',
   'app/data_sources/openaq_client.py',
   'app/data_sources/open_meteo_client.py',
   'app/data/validation.py'],
  'reusable_functions': ['_normalize_hourly_results',
   '_normalize_hourly_weather',
   '_validate_hourly_weather',
   '_normalize_utc_timestamp'],
  'notebook_defined_logic': ['PM2.5 quality-rule application',
   'duplicate-hour handling',
   'canonical hourly timestamp construction',
   'PM2.5 and weather merge',
   'canonical dataset validation']},
 'feature_engineering': {'files': ['app/features/live_feature_builder.py',
   'notebooks/03_build_training_dataset.ipynb'],
  'reusable_functions': ['validate_hourly_timeline',
   'add_pm25_history_features',
   'add_wind_direction_features',
   'add_time_features',
   'build_reference_feature_table',
   'build_target_weather_fe

In [15]:
manual_checks = {
    "canonical_sources_identified": bool(
        repository_integration_report[
            "canonical_pipeline"
        ]["files"]
    ),
    "canonical_notebook_logic_identified": bool(
        repository_integration_report[
            "canonical_pipeline"
        ].get(
            "notebook_defined_logic"
        )
    ),
    "feature_sources_identified": bool(
        repository_integration_report[
            "feature_engineering"
        ]["files"]
    ),
    "reusable_live_features_identified": bool(
        repository_integration_report[
            "feature_engineering"
        ]["reusable_functions"]
    ),
    "training_notebook_logic_identified": bool(
        repository_integration_report[
            "feature_engineering"
        ].get(
            "notebook_defined_logic"
        )
    ),
    "inference_feature_builder_identified": bool(
        repository_integration_report[
            "live_inference"
        ]["feature_builder_function"]
    ),
    "model_loader_identified": bool(
        repository_integration_report[
            "live_inference"
        ]["model_loader_function"]
    ),
    "feature_contract_validation_confirmed": (
        repository_integration_report[
            "live_inference"
        ]["feature_contract_validation"]
    ),
    "required_extractions_documented": bool(
        repository_integration_report[
            "small_changes_required"
        ]
    ),
}

In [16]:
automated_checks = {
    "all_required_artifacts_exist": (
        all(artifact_existence.values())
    ),
    "training_dataset_not_empty": (
        len(training_df) > 0
    ),
    "horizon_starts_at_1": (
        training_dataset_summary[
            "horizon_min"
        ]
        == 1
    ),
    "horizon_ends_at_72": (
        training_dataset_summary[
            "horizon_max"
        ]
        == 72
    ),
    "no_duplicate_training_keys": (
        training_dataset_summary[
            "duplicate_key_count"
        ]
        == 0
    ),
    "model_loaded_successfully": (
        approved_model_summary[
            "loaded_successfully"
        ]
    ),
    "model_checksum_available": bool(
        approved_model_summary[
            "checksum_sha256"
        ]
    ),
}

automated_checks

{'all_required_artifacts_exist': True,
 'training_dataset_not_empty': True,
 'horizon_starts_at_1': True,
 'horizon_ends_at_72': True,
 'no_duplicate_training_keys': True,
 'model_loaded_successfully': True,
 'model_checksum_available': True}

In [17]:
manual_checks = {
    "canonical_files_identified": bool(
        repository_integration_report[
            "canonical_pipeline"
        ][
            "files"
        ]
    ),
    "canonical_functions_identified": bool(
        repository_integration_report[
            "canonical_pipeline"
        ][
            "reusable_functions"
        ]
    ),
    "feature_files_identified": bool(
        repository_integration_report[
            "feature_engineering"
        ][
            "files"
        ]
    ),
    "feature_functions_identified": bool(
        repository_integration_report[
            "feature_engineering"
        ][
            "reusable_functions"
        ]
    ),
    "inference_feature_builder_identified": bool(
        repository_integration_report[
            "live_inference"
        ][
            "feature_builder_function"
        ]
    ),
    "model_loader_identified": bool(
        repository_integration_report[
            "live_inference"
        ][
            "model_loader_function"
        ]
    ),
    "feature_contract_validation_confirmed": (
        repository_integration_report[
            "live_inference"
        ][
            "feature_contract_validation"
        ]
    ),
}


phase_9a_approved = (
    all(automated_checks.values())
    and all(manual_checks.values())
)


repository_integration_report[
    "phase_9a_approved"
] = phase_9a_approved

repository_integration_report[
    "status"
] = (
    "REPOSITORY_INSPECTION_COMPLETED"
    if phase_9a_approved
    else "REPOSITORY_INSPECTION_INCOMPLETE"
)


REPORT_PATH = (
    REPORT_DIRECTORY
    / "repository_integration_report.json"
)


REPORT_PATH.write_text(
    json.dumps(
        repository_integration_report,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)


print("Phase 9A approved:", phase_9a_approved)
print("Report saved:", REPORT_PATH)

Phase 9A approved: True
Report saved: /home/riyan/Riyan/projects/pearls-aqi-predictor/reports/phase_9/repository_integration_report.json


## **9B.** Secure connectivity and dry-run configuration

Phase 9B establishes the MLOps integration boundary before any feature groups
or models are written to Hopsworks.

The implementation supports two independent backend selections:

- `FEATURE_STORE_BACKEND`
- `MODEL_REGISTRY_BACKEND`

Each backend can use either `local` or `hopsworks`.

Local mode preserves the existing Parquet and joblib workflow. Hopsworks mode
uses environment-based credentials and resolves the project's Feature Store
and Model Registry.

Dry-run mode remains enabled by default. A successful connection does not
create feature groups, insert data, or register models during this subphase.

In [18]:
from importlib.metadata import (
    PackageNotFoundError,
    version,
)

from app.mlops.config import (
    get_mlops_settings,
)


try:
    installed_hopsworks_version = version(
        "hopsworks"
    )
except PackageNotFoundError:
    installed_hopsworks_version = None


get_mlops_settings.cache_clear()
mlops_settings = get_mlops_settings()

print(
    "Hopsworks SDK version:",
    installed_hopsworks_version,
)

mlops_settings.safe_summary()

Hopsworks SDK version: 5.0.3


{'feature_store_backend': 'hopsworks',
 'model_registry_backend': 'hopsworks',
 'dry_run': False,
 'hopsworks_project': 'perarls_aqi_predictor',
 'hopsworks_host': 'eu-west.cloud.hopsworks.ai',
 'hopsworks_port': 443,
 'hopsworks_engine': 'python',
 'hostname_verification': True,
 'feature_group_version': 1,
 'feature_view_version': 2,
 'model_name': 'pearls_aqi_pm25_forecaster',
 'api_key_configured': True,
 'pm25_feature_group_name': 'pm25_hourly_observations',
 'weather_feature_group_name': 'weather_hourly_observations',
 'engineered_feature_group_name': 'pm25_hourly_features',
 'feature_pipeline_version': 'phase_2_v1',
 'source_data_version': 'phase_1_v1',
 'phase_1_canonical_dataset_path': 'data/processed/pearls_aqi_canonical_hourly.parquet',
 'phase_2_training_dataset_path': 'data/training/feature_dataset_full.parquet',
 'feature_view_name': 'pm25_reference_features',
 'training_dataset_name': 'pm25_72h_training_dataset',
 'training_dataset_version': 1,
 'training_dataset_float_t

### Local fallback validation

Local mode must remain usable without Hopsworks credentials.

This confirms that:

- existing Parquet artifacts remain available
- the local model artifact remains available
- Hopsworks credentials are not required
- no remote connection is attempted
- dry-run mode remains enabled

In [ ]:
from app.mlops.config import MLOpsSettings


local_settings = MLOpsSettings(
    feature_store_backend="local",
    model_registry_backend="local",
    mlops_dry_run=True,
)

assert not local_settings.uses_hopsworks
assert local_settings.mlops_dry_run
assert local_settings.hopsworks_api_key is None

local_settings.safe_summary()

### Hopsworks connection validation

This check authenticates with Hopsworks and resolves:

- the configured project
- the project's Feature Store
- the project's Model Registry
- the installed SDK version

The check is read-only. It does not create feature groups, insert records,
generate training datasets, or register models.

In [ ]:
from app.mlops.client import (
    connect_to_hopsworks,
)
from app.mlops.config import (
    get_mlops_settings,
)


get_mlops_settings.cache_clear()
settings = get_mlops_settings()

resources = connect_to_hopsworks(
    settings
)

connection_summary = {
    "connected": True,
    "sdk_version": resources.sdk_version,
    "project_name": resources.project_name,
    "feature_store_resolved": (
        resources.feature_store is not None
    ),
    "feature_store_name": (
        resources.feature_store_name
    ),
    "model_registry_resolved": (
        resources.model_registry is not None
    ),
    "engine": settings.hopsworks_engine,
    "dry_run": settings.mlops_dry_run,
}

connection_summary

2026-07-30 12:24:01,845 INFO: Initializing external client
2026-07-30 12:24:01,846 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-07-30 12:24:05,408 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/41128


{'connected': True,
 'sdk_version': '5.0.3',
 'project_name': 'perarls_aqi_predictor',
 'feature_store_resolved': True,
 'feature_store_name': 'perarls_aqi_predictor_featurestore',
 'model_registry_resolved': True,
 'engine': 'python',
 'dry_run': False}

## **9C.** Feature-store data contracts

Phase 9C defines the minimum practical Hopsworks feature-store model.

Three offline feature groups are required:

1. validated hourly PM2.5 observations
2. validated hourly historical weather observations
3. reusable reference-time engineered features

The design intentionally excludes:

- target PM2.5 labels from reusable feature groups
- train, validation, and test split markers
- complete horizon-expanded training rows
- online serving for all 72 forecast horizons
- weather forecast snapshots during the first migration

Stable entity identifiers are used instead of latitude and longitude as
primary keys. All event times are timezone-aware UTC timestamps.

Dry-run validation checks the contracts without creating or modifying remote
Hopsworks resources.

In [20]:
from app.mlops.config import (
    get_mlops_settings,
)
from app.mlops.contracts import (
    build_feature_group_contracts,
)
from app.mlops.feature_groups import (
    validate_contracts,
)


get_mlops_settings.cache_clear()
settings = get_mlops_settings()


contracts = build_feature_group_contracts(
    pm25_version=(
        settings.hopsworks_pm25_feature_group_version
    ),
    weather_version=(
        settings.hopsworks_weather_feature_group_version
    ),
    engineered_version=(
        settings.hopsworks_engineered_feature_group_version
    ),
    pm25_name=(
        settings.hopsworks_pm25_feature_group_name
    ),
    weather_name=(
        settings.hopsworks_weather_feature_group_name
    ),
    engineered_name=(
        settings.hopsworks_engineered_feature_group_name
    ),
    model_feature_columns=model_feature_columns,
)


validate_contracts(contracts)

contract_summaries = {
    key: contract.safe_summary()
    for key, contract in contracts.items()
}

contract_summaries

{'pm25': {'name': 'pm25_hourly_observations',
  'version': 1,
  'description': 'Validated hourly PM2.5 observations for the Zafar Memon DHA reference location.',
  'primary_key': ['location_key', 'sensor_id'],
  'event_time': 'datetime_utc',
  'online_enabled': False,
  'feature_count': 11,
  'features': [{'name': 'location_key',
    'offline_type': 'string',
    'description': 'Stable project location identifier.',
    'nullable': False},
   {'name': 'location_id',
    'offline_type': 'bigint',
    'description': 'OpenAQ location identifier.',
    'nullable': False},
   {'name': 'sensor_id',
    'offline_type': 'bigint',
    'description': 'OpenAQ sensor identifier.',
    'nullable': False},
   {'name': 'datetime_utc',
    'offline_type': 'timestamp',
    'description': 'UTC hour represented by the observation.',
    'nullable': False},
   {'name': 'pm25_ug_m3',
    'offline_type': 'double',
    'description': 'Validated hourly PM2.5 concentration.',
    'nullable': True},
   {'name':

### Feature-group boundaries

The PM2.5 and weather feature groups preserve validated source observations.

The engineered feature group contains only features available at the
reference time. It excludes labels, target timestamps, and future PM2.5
information.

Target-hour weather and the 1–72 horizon expansion remain part of the
controlled training-dataset and live-inference builders. This avoids forcing
the full prediction matrix into an online feature store.

In [ ]:
pm25_contract = contracts["pm25"]
weather_contract = contracts["weather"]
engineered_contract = contracts[
    "engineered"
]


assert pm25_contract.event_time == (
    "datetime_utc"
)

assert weather_contract.event_time == (
    "datetime_utc"
)

assert engineered_contract.event_time == (
    "reference_time"
)

assert not pm25_contract.online_enabled
assert not weather_contract.online_enabled
assert not engineered_contract.online_enabled

assert "target_pm25_ug_m3" not in (
    engineered_contract.feature_names
)

assert "target_time" not in (
    engineered_contract.feature_names
)

assert "forecast_horizon_hours" not in (
    engineered_contract.feature_names
)

print(
    "Phase 9C feature-group contracts validated."
)

Phase 9C feature-group contracts validated.


### Dry-run feature-group validation

Dry-run mode validates:

- feature-group names and versions
- primary-key columns
- event-time columns
- unique feature names
- exclusion of target leakage
- local model-feature contract compatibility

No remote feature-group metadata or data is written.

In [ ]:
from app.mlops.client import (
    connect_to_hopsworks,
)
from app.mlops.feature_groups import (
    create_or_get_feature_groups,
)


resources = connect_to_hopsworks(
    settings
)

resolved_feature_groups = (
    create_or_get_feature_groups(
        resources=resources,
        settings=settings,
        contracts=contracts,
    )
)

resolved_feature_groups.safe_summary()

2026-07-30 12:49:01,635 INFO: Closing external client and cleaning up certificates.
2026-07-30 12:49:01,639 INFO: Connection closed.
2026-07-30 12:49:01,643 INFO: Initializing external client
2026-07-30 12:49:01,646 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-07-30 12:49:03,662 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/41128


{'dry_run': False,
 'pm25_resolved': True,
 'weather_resolved': True,
 'engineered_resolved': True}

In [ ]:
from datetime import datetime, timezone
import json


FEATURE_GROUP_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "phase_9"
    / "feature_group_validation_report.json"
)


feature_group_report = {
    "phase": "9C",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": (
        "FEATURE_GROUP_CONTRACTS_VALIDATED"
    ),
    "dry_run": settings.mlops_dry_run,
    "remote_writes_performed": False,
    "location_key": (
        "zafar_memon_dha_karachi"
    ),
    "feature_group_version": (
        settings.hopsworks_feature_group_version
    ),
    "contracts": contract_summaries,
    "required_feature_group_count": 3,
    "online_feature_groups_enabled": 0,
    "target_columns_excluded": True,
    "event_times_use_utc": True,
    "feature_group_schemas_validated": True,
    "phase_9c_approved": True,
}


FEATURE_GROUP_REPORT_PATH.write_text(
    json.dumps(
        feature_group_report,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Feature-group report saved:",
    FEATURE_GROUP_REPORT_PATH,
)

feature_group_report

Feature-group report saved: /home/riyan/Riyan/projects/pearls-aqi-predictor/reports/phase_9/feature_group_validation_report.json


{'phase': '9C',
 'generated_at_utc': '2026-07-30T07:24:14.513283+00:00',
 'status': 'FEATURE_GROUP_CONTRACTS_VALIDATED',
 'dry_run': False,
 'remote_writes_performed': False,
 'location_key': 'zafar_memon_dha_karachi',
 'feature_group_version': 1,
 'contracts': {'pm25': {'name': 'pm25_hourly_observations',
   'version': 1,
   'description': 'Validated hourly PM2.5 observations for the Zafar Memon DHA reference location.',
   'primary_key': ['location_key', 'sensor_id'],
   'event_time': 'datetime_utc',
   'online_enabled': False,
   'feature_count': 11,
   'features': [{'name': 'location_key',
     'offline_type': 'string',
     'description': 'Stable project location identifier.',
     'nullable': False},
    {'name': 'location_id',
     'offline_type': 'bigint',
     'description': 'OpenAQ location identifier.',
     'nullable': False},
    {'name': 'sensor_id',
     'offline_type': 'bigint',
     'description': 'OpenAQ sensor identifier.',
     'nullable': False},
    {'name': 'date

## **9D.** Historical backfill and gap detection

The first Hopsworks migration uses the already validated local artifacts rather
than downloading the full source period again.

The migration prepares:

- hourly PM2.5 observation rows from the canonical dataset
- hourly weather observation rows from the canonical dataset
- one reusable engineered feature row per reference timestamp from the
  validated Phase 2 dataset

The process is idempotent:

- new logical records are inserted
- changed logical records are updated
- unchanged records are skipped
- duplicate logical keys are rejected before writing

Dry-run mode performs all local preparation, schema validation, gap detection,
and row classification without writing to Hopsworks.

### Event-time and ingestion-time semantics

Each feature group distinguishes between:

- **event time** — the UTC hour represented by the observation or feature
- **ingestion time** — the time the historical migration prepared the row

The feature groups use:

| Feature group | Event time |
|---|---|
| PM2.5 observations | `datetime_utc` |
| Weather observations | `datetime_utc` |
| Engineered features | `reference_time` |

Hopsworks combines the entity primary key with event time in the offline store,
which preserves one historical value for each entity-hour.

In [ ]:
from pathlib import Path

from app.mlops.config import (
    get_mlops_settings,
)


get_mlops_settings.cache_clear()
settings = get_mlops_settings()

canonical_path = (
    PROJECT_ROOT
    / settings.phase_1_canonical_dataset_path
)

training_path = (
    PROJECT_ROOT
    / settings.phase_2_training_dataset_path
)

print("Canonical dataset:", canonical_path)
print("Canonical exists:", canonical_path.exists())

print()
print("Training dataset:", training_path)
print("Training exists:", training_path.exists())

assert canonical_path.exists()
assert training_path.exists()
assert settings.mlops_dry_run

In [ ]:
import json


DRY_RUN_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "phase_9"
    / "historical_backfill_dry_run_report.json"
)

dry_run_report = json.loads(
    DRY_RUN_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)

dry_run_report

{'phase': '9D',
 'pipeline_run_id': 'historical_backfill_3ddc90740cc64fea8d675c0b3eb263fc',
 'pipeline_name': 'historical_feature_backfill',
 'status': 'BACKFILL_DRY_RUN_SUCCESS',
 'started_at_utc': '2026-07-30T07:21:26.902019+00:00',
 'completed_at_utc': '2026-07-30T07:21:34.465713+00:00',
 'start_time_utc': '2025-07-08T00:00:00+00:00',
 'end_time_utc': '2026-07-23T23:00:00+00:00',
 'dry_run': True,
 'remote_writes_performed': False,
 'canonical_dataset_path': 'data/processed/pearls_aqi_canonical_hourly.parquet',
 'training_dataset_path': 'data/training/feature_dataset_full.parquet',
 'feature_group_version': 1,
 'groups': {'pm25': {'feature_group_name': 'pm25_hourly_observations',
   'version': 1,
   'candidate_rows': 9144,
   'existing_rows_in_range': 0,
   'rows_to_insert': 9144,
   'rows_to_update': 0,
   'rows_unchanged': 0,
   'rows_written': 0,
   'duplicate_keys': 0,
   'missing_interval_count': 0,
   'missing_intervals': []},
  'weather': {'feature_group_name': 'weather_hourl

In [ ]:
from app.mlops.client import (
    connect_to_hopsworks,
)
from app.mlops.config import (
    get_mlops_settings,
)


get_mlops_settings.cache_clear()
settings = get_mlops_settings()

assert not settings.mlops_dry_run

resources = connect_to_hopsworks(
    settings
)

feature_store = resources.feature_store

pm25_group = feature_store.get_feature_group(
    name=(
        settings
        .hopsworks_pm25_feature_group_name
    ),
    version=(
        settings
        .hopsworks_feature_group_version
    ),
)

weather_group = feature_store.get_feature_group(
    name=(
        settings
        .hopsworks_weather_feature_group_name
    ),
    version=(
        settings
        .hopsworks_feature_group_version
    ),
)

engineered_group = (
    feature_store.get_feature_group(
        name=(
            settings
            .hopsworks_engineered_feature_group_name
        ),
        version=(
            settings
            .hopsworks_feature_group_version
        ),
    )
)

start_time = pd.Timestamp(
    "2025-07-08T00:00:00Z"
)

end_exclusive = pd.Timestamp(
    "2026-07-24T00:00:00Z"
)

pm25_readback = pm25_group.read(
    dataframe_type="pandas",
    start_time=start_time.to_pydatetime(),
    end_time=end_exclusive.to_pydatetime(),
)

weather_readback = weather_group.read(
    dataframe_type="pandas",
    start_time=start_time.to_pydatetime(),
    end_time=end_exclusive.to_pydatetime(),
)

engineered_readback = engineered_group.read(
    dataframe_type="pandas",
    start_time=start_time.to_pydatetime(),
    end_time=end_exclusive.to_pydatetime(),
)

readback_summary = {
    "pm25_rows": len(pm25_readback),
    "weather_rows": len(
        weather_readback
    ),
    "engineered_rows": len(
        engineered_readback
    ),
}

readback_summary

2026-07-30 12:51:48,142 INFO: Closing external client and cleaning up certificates.
2026-07-30 12:51:48,151 INFO: Connection closed.
2026-07-30 12:51:48,153 INFO: Initializing external client
2026-07-30 12:51:48,155 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443


2026-07-30 12:51:50,912 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/41128
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (10.36s) 
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (10.73s) 
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (13.84s) 


{'pm25_rows': 9144, 'weather_rows': 9144, 'engineered_rows': 7321}

## **9E.** Feature view and training-dataset parity

Phase 9E creates a reusable feature view over the engineered reference-time
feature group.

The complete 72-horizon training table is not stored as another feature group.
Instead, it is reconstructed from:

- reusable reference-time features read from Hopsworks
- the approved Phase 2 horizon and target structure
- target-hour historical weather
- the PM2.5 training label

This migration approach preserves the validated Phase 2 behavior while
confirming that the Hopsworks reference-time features match the local training
dataset.

The generated dataset must match the approved Phase 2 dataset in:

- row count
- column names and order
- reference timestamps
- forecast horizons
- target timestamps
- target values
- feature values
- duplicate-key count

### Historical target-weather limitation

Historical training uses observed target-hour weather as a proxy for weather
forecasts.

Production inference uses Open-Meteo forecast weather. This limitation remains
documented and is not treated as complete training-serving equivalence.

Future weather-forecast snapshots may be stored for newer model versions.

In [21]:
import json


TRAINING_DATASET_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "phase_9"
    / "training_dataset_report.json"
)

training_dataset_report = json.loads(
    TRAINING_DATASET_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)

training_dataset_report

{'phase': '9E',
 'generated_at_utc': '2026-07-30T10:07:58.439557+00:00',
 'status': 'TRAINING_DATASET_PARITY_PASSED',
 'feature_view': {'name': 'pm25_reference_features',
  'version': 2,
  'dry_run': False,
  'resolved': True,
  'feature_count': 38},
 'training_dataset': {'name': 'pm25_72h_training_dataset',
  'version': 1,
  'snapshot_path': 'data/training/hopsworks/pm25_72h_training_dataset_v1.parquet',
  'row_count': 512867,
  'column_count': 59,
  'reference_start': '2025-07-09 00:00:00+00:00',
  'reference_end': '2026-07-23 22:00:00+00:00',
  'horizon_min': 1,
  'horizon_max': 72},
 'parity': {'passed': True,
  'local_rows': 512867,
  'generated_rows': 512867,
  'local_columns': 59,
  'generated_columns': 59,
  'duplicate_keys': 0,
  'missing_columns': [],
  'additional_columns': [],
  'mismatched_columns': [],
  'maximum_numeric_difference': 0.0},
 'historical_weather_limitation': 'Observed target-hour historical weather remains a proxy for forecast weather.'}